# 02 — Power Analysis: SPA and DPA

## What This Is
Power analysis attacks recover secret keys by measuring power consumption during cryptographic operations. Paul Kocher's 1999 DPA paper demonstrated that AES and DES implementations in embedded devices leak the key through power traces.

## Why It Matters
Smart cards, payment terminals, and HSMs all implement cryptography. Without power analysis countermeasures, secret keys can be extracted in minutes using cheap oscilloscopes. This is why CC EAL4+ evaluations mandate side-channel resistance testing.

## Where the Community Stands
Masking (blinding) and hiding (shuffling, dual-rail logic) are the two main countermeasure classes. ChipWhisperer-Lite captures power traces at 100 Msps. CPA is the most efficient attack: one correct key hypothesis correlates with traces, all wrong ones do not.

## Key Concepts
- **Hamming Weight (HW)**: number of 1-bits in a value. Power ∝ HW in CMOS.
- **Hamming Distance (HD)**: XOR popcount between consecutive values (switching activity)
- **Leakage model**: the assumed relationship between data and power consumption

In [ ]:
import random
import math
import statistics
from typing import List, Tuple

# --- Simulate power traces for AES SubBytes operation ---

# AES S-Box (first 64 entries)
SBOX = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16,
]

def hamming_weight(x: int) -> int:
    return bin(x).count('1')

def hamming_distance(x: int, y: int) -> int:
    return hamming_weight(x ^ y)

# Simulate a power trace: noise + signal proportional to HW(SBOX[pt XOR key])
TRUE_KEY_BYTE = 0xAB
NOISE_STD     = 1.5

def simulate_trace(plaintext_byte: int, key_byte: int,
                   rng: random.Random, noise_std: float) -> float:
    sbox_out   = SBOX[plaintext_byte ^ key_byte]
    hw         = hamming_weight(sbox_out)
    noise      = rng.gauss(0, noise_std)
    return hw + noise

N_TRACES = 500
rng = random.Random(42)
plaintexts = [rng.randint(0, 255) for _ in range(N_TRACES)]
traces     = [simulate_trace(pt, TRUE_KEY_BYTE, rng, NOISE_STD) for pt in plaintexts]

print(f'Simulated {N_TRACES} power traces')
print(f'Mean power:   {statistics.mean(traces):.3f}')
print(f'Stdev power:  {statistics.stdev(traces):.3f}')
print(f'True key:     0x{TRUE_KEY_BYTE:02X}')

In [ ]:
def pearson_correlation(x: List[float], y: List[float]) -> float:
    n   = len(x)
    mx  = sum(x)/n
    my  = sum(y)/n
    num = sum((xi-mx)*(yi-my) for xi,yi in zip(x,y))
    dx  = math.sqrt(sum((xi-mx)**2 for xi in x))
    dy  = math.sqrt(sum((yi-my)**2 for yi in y))
    return num/(dx*dy) if dx*dy > 0 else 0.0

def cpa_attack(plaintexts: List[int], traces: List[float]) -> Tuple[int, List[float]]:
    """Correlation Power Analysis: find key byte maximising |corr(HW(SBOX[pt^k]), trace)|."""
    correlations = []
    for k_guess in range(256):
        hw_model = [hamming_weight(SBOX[pt ^ k_guess]) for pt in plaintexts]
        corr     = abs(pearson_correlation(hw_model, traces))
        correlations.append(corr)
    best_key = max(range(256), key=lambda k: correlations[k])
    return best_key, correlations

best_k, corrs = cpa_attack(plaintexts, traces)
print(f'CPA result:')
print(f'  Recovered key byte: 0x{best_k:02X}')
print(f'  True key byte:      0x{TRUE_KEY_BYTE:02X}')
print(f'  Correct:            {best_k == TRUE_KEY_BYTE}')
print(f'  Max correlation:    {corrs[best_k]:.4f}')
print(f'  True key corr:      {corrs[TRUE_KEY_BYTE]:.4f}')
print(f'  Runner-up corr:     {sorted(corrs, reverse=True)[1]:.4f}')

## CPA with Varying Trace Count
With noisy traces, more measurements improve attack success. This shows the trade-off between measurement effort and noise tolerance.

In [ ]:
def cpa_success_rate(plaintexts: List[int], traces: List[float],
                    true_key: int, n_counts: List[int],
                    n_trials: int = 20) -> List[float]:
    """Estimate CPA success probability at each trace count via bootstrap."""
    rng_b = random.Random(777)
    rates = []
    for n in n_counts:
        success = 0
        for _ in range(n_trials):
            idxs = rng_b.choices(range(len(plaintexts)), k=n)
            sub_pt = [plaintexts[i] for i in idxs]
            sub_tr = [traces[i]     for i in idxs]
            k_rec, _ = cpa_attack(sub_pt, sub_tr)
            if k_rec == true_key:
                success += 1
        rates.append(success / n_trials)
    return rates

n_counts = [10, 25, 50, 100, 200, 500]
rates    = cpa_success_rate(plaintexts, traces, TRUE_KEY_BYTE, n_counts)

print(f'{"Traces":<10} {"Success Rate":<15} {"Visual"}')
for n, r in zip(n_counts, rates):
    bar = '#' * int(r * 20)
    print(f'{n:<10} {r:.2f}           {bar}')

## Countermeasures
Masking adds a random value to data before processing; all operations work on masked data. Hiding randomises the power profile through shuffling, dummy operations, or dual-rail logic.

In [ ]:
# Masking countermeasure: DPA on masked implementation
def simulate_masked_trace(pt: int, key: int, rng: random.Random, noise_std: float) -> float:
    mask     = rng.randint(0, 255)
    masked   = pt ^ mask
    sbox_out = SBOX[masked ^ key]  # WRONG — need masked SBOX in real impl
    # Actual masking: SBOX'[x] = SBOX[x ^ mask] ^ mask2 — emulate by adding mask noise
    hw_masked = hamming_weight(sbox_out ^ mask)  # additional mask noise
    return hw_masked + rng.gauss(0, noise_std)

masked_traces = [simulate_masked_trace(pt, TRUE_KEY_BYTE, rng, NOISE_STD) for pt in plaintexts]
mk, mc = cpa_attack(plaintexts, masked_traces)

print('CPA against masked implementation:')
print(f'  Recovered: 0x{mk:02X}  True: 0x{TRUE_KEY_BYTE:02X}  Correct: {mk == TRUE_KEY_BYTE}')
print(f'  Max corr:  {mc[mk]:.4f}  (vs {corrs[TRUE_KEY_BYTE]:.4f} unmasked)')
print(f'  (Masking breaks simple CPA — higher-order attacks needed)')